# Validation for a new model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import numpy as np
import numpy.typing as npt
import xarray as xr
from model_plots import (
    plot_et_comparison,
    plot_pixel_evolution,
    plot_variable_evolution,
    print_error_metrics,
)
from model_tools import (
    et_sd_list,
    extra_variable_list,
    find_directories,
    get_et_single_date,
    radiation_list,
)


## Input Data

We obtained the observed evapotranspiration, daily radiation, and extra ERA5-Land variables using the **prepare_daily_et_timeseries**, **prepare_daily_radiation_timeseries** functions from **ET-DATASET**, respectively. The data were then stored for each region of interest in a dedicated subdirectory within **SWEAT_TEST_DATA_PATH**.

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

In [ ]:
input_dir = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "senegal"
)
dir_sd, dir_rad, dir_var = find_directories(input_dir)

Time period

In [ ]:
start_date = "2024-03-01"
end_date = "2024-03-31"

Location

In [ ]:
x = 345700
y = 1590936

Plot data

In [ ]:
data = get_et_single_date(dir_sd, dt.datetime.strptime(start_date, "%Y-%m-%d"))
data.plot(cmap="viridis")

Plot data for a pixel

In [ ]:
plot_variable_evolution(dir_var, dir_sd, dir_rad, start_date, end_date, x, y)

Extract a timeseries for one pixel

In [ ]:
time = xr.date_range(start_date, end_date)
tp = np.array(extra_variable_list(dir_var, "tp", start_date, end_date, x, y))
sw = np.array(extra_variable_list(dir_var, "sw", start_date, end_date, x, y))
sro = np.array(extra_variable_list(dir_var, "sro", start_date, end_date, x, y))
src = np.array(extra_variable_list(dir_var, "src", start_date, end_date, x, y))
et_sd = np.array(et_sd_list(dir_sd, start_date, end_date, x, y))
rad = np.array(radiation_list(dir_rad, start_date, end_date, x, y))

In [ ]:
plot_pixel_evolution(time, et_sd, rad, tp, sw)

Mask data

In [ ]:
def mask_data(data: npt.NDArray, freq=tuple[int, int]) -> npt.NDArray:
    """
    Mask data

    Parameters
    ----------
    data: np.array
        Data
    freq: tuple(int,int)
        Frequency interval

    Return
    ------
    masked_data: np.array
        Masked data
    """
    masked_data = np.nan * data
    i = 0
    masked_data[i] = data[i]
    while i < data.size:
        step = np.random.randint(freq[0], freq[1] + 1)
        i += step
        if i < data.size:
            masked_data[i] = data[i]
    return masked_data

In [ ]:
masked_et_sd = mask_data(et_sd, freq=(3, 3))

### Test on a dummy time series generation algorithm.

In [ ]:
# dummy algorithm
def dummy_et(
    et: npt.NDArray,
    rad: npt.NDArray,  # noqa
    tp: npt.NDArray,  # noqa
) -> npt.NDArray:
    """
    Computes the simulated daily evapotranspiration at a given pixel
    over a specified period using a dummy algorithm (mean of all variables)

    Parameters
    ----------
    et: np.array
        Daily ET timeseries with missing values
    rad: np.array
        Daily radiation timeseries
    tp: np.array
        Daily precipitation timeseries

    Returns
    -------
    et: xr.DataArray
        Daily ET timeseries
    """
    mean = np.nanmean(et)
    return np.nan_to_num(et, nan=mean)

In [ ]:
# time series generation
et_ts = dummy_et(masked_et_sd, rad, tp)

### Results

In [ ]:
plot_et_comparison(et_sd, et_ts, start_date, end_date)


In [ ]:
print_error_metrics(et_sd, et_ts, 3)